# 04 — Train: SARIMAX

Fits one fixed-order SARIMAX model with a deliberately minimal daily seasonal AR term using the six calendar Fourier terms as exogenous regressors for the configured target station and evaluates it once on the test feature artifact. A local ADF/ACF/PACF diagnostic on the train water-level series found negligible daily autocorrelation after detrending, so the seasonal order tests for a weak daily effect rather than assuming a large one. Weather and water-level lag/rolling features are excluded from exog because multi-step forecasting needs their values at every future step, and future weather is never known in advance.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview and test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins the notebook's constants: the fixed SARIMAX order, the exogenous regressor columns, the artifact paths and the Stage-3 column contract.

**What the imports provide**

- `SARIMAX` — statsmodels' seasonal ARIMA with exogenous regressors, estimated by maximum likelihood through a Kalman filter. The filter is why the model can absorb a new observation without refitting, which the rolling test loop below depends on.
- `math`, `numpy` — used by `calendar_exog()` to rebuild the Fourier terms for arbitrary future timestamps.
- `mean_absolute_error`, `root_mean_squared_error` — the two reported error metrics.
- `DEFAULT_FEATURE_CONFIG`, `feature_column_names()`, `target_column_names()` — the Stage-3 contract, so the notebook cannot drift away from the artifacts it reads.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed` | Directory the Stage-3 feature Parquets are read from. This notebook only reads — it never writes back. |
| `PREDICTION_PREVIEW_ROWS` | `5` | How many scored test rows the final preview table shows. Display only; it has no effect on any metric. |
| `FEATURE_COLUMNS` | 53 names | The frozen Stage-3 predictor contract, read from `feature_column_names()` instead of being hardcoded so the notebook fails loudly if Stage 3 ever changes it. It contains: the raw `water_level`, `imputed`, `precipitation` and `temperature_2m`; 8 water-level lags (1, 3, 6, 12, 24, 48, 72, 168 h); 5 water-level differences (1–24 h); 16 rolling water-level statistics (mean/std/min/max x 6/24/72/168 h); 4 rolling imputation counts; 4 rolling precipitation sums; 4 rolling temperature means plus the 24 h temperature min and max; and 6 calendar Fourier terms. |
| `TARGET_COLUMNS` | `target_t_plus_01` … `target_t_plus_24` | The 24 future water levels, one per lead hour. The model emits all of them from a single feature vector — a *direct* multi-horizon setup, with no recursive feeding of its own predictions. |
| `SARIMAX_ORDER` | `(1, 1, 1)` | The non-seasonal `(p, d, q)`. `p=1`: one autoregressive lag, so this hour's level depends on the previous hour's. `d=1`: fit on first differences rather than levels, because a river level wanders without returning to a fixed mean and is not stationary as-is. `q=1`: one moving-average term, letting the model carry over the previous step's shock. `(1, 1, 1)` is the standard minimal choice — hand-picked, not searched. |
| `SARIMAX_SEASONAL_ORDER` | `(1, 0, 0, 24)` | The seasonal `(P, D, Q, m)` at period `m=24` (one day of hourly data). `P=1`: a single autoregressive term at lag 24. `D=0`: no seasonal differencing — the daily pattern is not strong enough to warrant removing it outright. `Q=0`: no seasonal moving-average term. This is deliberately the smallest seasonal block that can exist: the ADF/ACF/PACF diagnostic found only negligible daily autocorrelation after detrending, so the notebook spends exactly one parameter testing for a weak daily effect instead of assuming a large one. Each additional seasonal term at lag 24 is expensive, because it widens the state space by roughly the seasonal period. |
| `CALENDAR_EXOG_COLUMNS` | 6 names | The exogenous regressors: sine and cosine pairs for hour-of-day, day-of-week and day-of-year. Each cycle needs both a sine and a cosine so the encoding is continuous across the wrap-around (hour 23 sits next to hour 0, not 23 units away from it), and so a phase shift can be expressed as a linear combination of the two. |
| `CALENDAR_TIMEZONE` | `DEFAULT_FEATURE_CONFIG.calendar_timezone` (UTC) | Read from the Stage-3 config rather than hardcoded, so `calendar_exog()` reproduces the artifact's calendar columns exactly instead of assuming a zone. |
| `TRAIN_WATER_LEVEL_INTERPOLATION` | `linear`, `inside` | Settings for the temporary gap-filling applied to the *training input only*. `method="linear"` draws a straight line between the observations bracketing a gap. `limit_area="inside"` restricts that to gaps with an observation on **both** sides, so nothing is ever extrapolated past the first or last observation. The third key, `scope`, is documentation carried into the displayed table — it is not passed to pandas. |

**Why only the calendar columns are used as exogenous regressors**

`get_forecast` demands a value for every exogenous regressor at every one of the 24 future steps. Weather, lags and rolling statistics simply do not exist for those steps at issue time — using them would mean either inventing them or reading the future. The six calendar terms are the only predictors that are genuinely known in advance: they are a deterministic function of the clock and can be computed for any timestamp, forever.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

from src.config import FORECAST_HORIZON_HOURS, TARGET_STATION_ID
from src.feature_engineering import (
    DEFAULT_FEATURE_CONFIG,
    feature_column_names,
    target_column_names,
)

PROCESSED_DIR = Path("data/processed")
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())
CALENDAR_EXOG_COLUMNS = [
    "utc_hour_sin",
    "utc_hour_cos",
    "utc_day_of_week_sin",
    "utc_day_of_week_cos",
    "utc_day_of_year_sin",
    "utc_day_of_year_cos",
]
CALENDAR_TIMEZONE = DEFAULT_FEATURE_CONFIG.calendar_timezone
TRAIN_WATER_LEVEL_INTERPOLATION = {
    "method": "linear",
    "limit_area": "inside",
    "scope": "training input passed to SARIMAX only",
}
SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEASONAL_ORDER = (1, 0, 0, 24)

## Shared evaluation cohort

SARIMAX sees only two things: the water-level series as its endogenous variable, and the six calendar Fourier terms as exogenous regressors. It is nevertheless scored on exactly the same cohort as Ridge, the tree models and the persistence baseline — issue times that Stage 3 marked `target_valid` (all 24 future hours genuinely observed) **and** whose full 53-column predictor vector is present.

The predictors the model does not consume are therefore never handed to it; they exist here purely to define a comparable cohort. This costs some rows the model could technically have scored, and buys the ability to put this notebook's MAE next to the tree models' MAE without an asterisk.

## Helper functions

Five helpers used by the cells below. The keyword-only `station_id` / `artifact_name` arguments exist so that any raised error names the split it came from.

**`eligible_rows(frame, *, station_id, artifact_name) -> pd.Series`** — returns the boolean cohort mask described above, and raises if the artifact is missing a contract column or carries a null target inside a `target_valid` row.

**`water_level_series(frame, *, station_id, artifact_name) -> np.ndarray`** — returns the split's water level as a plain float array, but only after proving the series is safe for a state-space model: single station, timezone-aware UTC timestamps, no duplicates, a contiguous hourly grid, numeric values, no infinities. `NaN` is permitted and preserved.

**`interpolated_train_water_levels(values) -> (prepared, missing_rows)`**

- `values` — the raw training water-level array, gaps included.
- Returns the gap-filled array and the number of rows that were missing beforehand.

Filling is linear with `limit_area="inside"`, so only gaps with real observations on both sides are filled; a gap at either end cannot be filled and the helper raises instead of passing a `NaN` to the fit. This affects only the temporary array handed to SARIMAX — matching the input Auto-ARIMA receives, so the three notebooks stay comparable — and never the artifact on disk.

**`calendar_exog(timestamps) -> pd.DataFrame`**

- `timestamps` — any `DatetimeIndex`, including future hours that appear nowhere in the artifacts.

Recomputes the six Fourier columns from scratch: the timestamps are converted to `CALENDAR_TIMEZONE`, then each cycle is turned into an angle (`2*pi*hour/24`, `2*pi*weekday/7`, `2*pi*(dayofyear-1)/days_in_year`, with `days_in_year` switching to 366 in leap years) and emitted as a sine/cosine pair. This function is what makes multi-step forecasting possible at all — see the verification cell below.

**`metric_tables(actual, predictions, *, station_id)`** and **`prediction_preview(frame, predictions)`** — identical to the other stage-4 notebooks. `metric_tables` returns one aggregate MAE/RMSE over all `n x 24` values plus the same pair per lead hour, which is what shows how fast accuracy decays from `t+1` to `t+24`. RMSE is always at least MAE and is dominated by the worst misses, so a wide gap between them points to a few large errors rather than uniformly poor accuracy.

In [ ]:
def eligible_rows(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(
        axis=1
    )
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible

In [ ]:
def water_level_series(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> np.ndarray:
    """Return one UTC-hourly station water-level series with no infinities."""
    required_columns = {"timestamp", "station_id", "water_level"}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")

    timestamp_dtype = frame["timestamp"].dtype
    if (
        not isinstance(timestamp_dtype, pd.DatetimeTZDtype)
        or str(timestamp_dtype.tz) != "UTC"
    ):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be timezone-aware UTC"
        )
    timestamps = pd.DatetimeIndex(frame["timestamp"])
    if timestamps.hasnans:
        raise ValueError(f"{station_id} {artifact_name} timestamps must be complete")
    expected_grid = pd.date_range(timestamps[0], periods=len(timestamps), freq="h")
    if timestamps.has_duplicates or not timestamps.equals(expected_grid):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be unique, ascending, and hourly"
        )

    station_ids = frame["station_id"].drop_duplicates().tolist()
    if station_ids != [station_id]:
        raise ValueError(
            f"{artifact_name} artifact must contain only station {station_id!r}; "
            f"got {station_ids!r}"
        )

    try:
        values = pd.to_numeric(frame["water_level"], errors="raise").to_numpy(
            dtype=float
        )
    except (TypeError, ValueError) as error:
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must be numeric"
        ) from error
    if np.isinf(values).any():
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must not contain infinities"
        )
    return values

In [ ]:
def interpolated_train_water_levels(values: np.ndarray) -> tuple[np.ndarray, int]:
    """Linearly fill internal training gaps for SARIMAX only."""
    missing_rows = int(np.isnan(values).sum())
    interpolated = pd.Series(values).interpolate(
        method=TRAIN_WATER_LEVEL_INTERPOLATION["method"],
        limit_area=TRAIN_WATER_LEVEL_INTERPOLATION["limit_area"],
    )
    prepared = interpolated.to_numpy(dtype=float)
    if not np.isfinite(prepared).all():
        raise ValueError(
            "SARIMAX training input has unfillable water_level gaps at a series boundary"
        )
    return prepared, missing_rows

In [ ]:
def calendar_exog(timestamps: pd.DatetimeIndex) -> pd.DataFrame:
    """Recompute the six calendar Fourier exog columns for arbitrary UTC timestamps."""
    converted = pd.DatetimeIndex(timestamps).tz_convert(CALENDAR_TIMEZONE)
    hour_angle = 2.0 * math.pi * converted.hour / 24.0
    weekday_angle = 2.0 * math.pi * converted.dayofweek / 7.0
    days_in_year = np.where(converted.is_leap_year, 366.0, 365.0)
    year_angle = 2.0 * math.pi * (converted.dayofyear - 1) / days_in_year
    return pd.DataFrame(
        {
            "utc_hour_sin": np.sin(hour_angle),
            "utc_hour_cos": np.cos(hour_angle),
            "utc_day_of_week_sin": np.sin(weekday_angle),
            "utc_day_of_week_cos": np.cos(weekday_angle),
            "utc_day_of_year_sin": np.sin(year_angle),
            "utc_day_of_year_cos": np.cos(year_angle),
        },
        index=timestamps,
    )

In [ ]:
def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
                "rmse": root_mean_squared_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[target], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon


def prediction_preview(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load and validate feature artifacts

Resolves `data/processed/<station>_train_features.parquet` and `<station>_test_features.parquet` for the station in `src.config.TARGET_STATION_ID`, raising `FileNotFoundError` if either is absent.

Then `water_level_series()` validates each split **independently**, because a state-space model consumes the raw series rather than a bag of rows, and a silently misaligned timeline would produce plausible-looking nonsense. The checks are that the artifact contains only the target station, that its timestamps are timezone-aware UTC, unique, ascending and form a contiguous hourly grid with no gaps, and that `water_level` is numeric and free of infinities.

`NaN` is explicitly allowed at this point. Values that Stage 2 filled in are flagged `imputed=True` but are real numbers; gaps too long to fill were deliberately preserved as missing, and they stay missing here. What happens to them differs by stage: in the training input they are interpolated (see below), while in the test loop they are appended to the filter as genuinely missing.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)
train_water_levels = water_level_series(
    train_features, station_id=station_id, artifact_name="train"
)
test_water_levels = water_level_series(
    test_features, station_id=station_id, artifact_name="test"
)

## Verify calendar exog reconstruction

`get_forecast` needs exogenous values for future steps, and those steps routinely fall past the last row of the test artifact — so they cannot be looked up, only recomputed. `calendar_exog()` does that recomputation, which means every forecast depends on it being *exactly* the function Stage 3 used.

This cell proves it before anything relies on it: recompute the six columns for the test artifact's own timestamps, where the true values are known, and compare them with `np.allclose` (which allows floating-point rounding but nothing else). Any mismatch — a timezone difference, a leap-year rule, an off-by-one in day-of-year — raises here rather than silently degrading every forecast downstream.

In [ ]:
reconstructed_test_exog = calendar_exog(pd.DatetimeIndex(test_features["timestamp"]))
if not np.allclose(
    reconstructed_test_exog[CALENDAR_EXOG_COLUMNS].to_numpy(),
    test_features[CALENDAR_EXOG_COLUMNS].to_numpy(),
):
    raise ValueError("calendar_exog() does not reproduce the artifact's calendar columns")

## Apply the eligibility cohort

Builds both masks, stops early if either split has no usable row, and keeps `test_rows` for scoring.

It also prepares `train_model_values`: the training series with its internal gaps linearly interpolated. This copy exists only in memory and only as SARIMAX's training input — the artifact on disk keeps its gaps. `interpolated_train_rows` records how many values were reconstructed and is displayed in the next cell, so the size of that intervention is visible rather than implied.

In [ ]:
train_mask = eligible_rows(train_features, station_id=station_id, artifact_name="train")
test_mask = eligible_rows(test_features, station_id=station_id, artifact_name="test")
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

test_rows = test_features.loc[test_mask]
train_model_values, interpolated_train_rows = interpolated_train_water_levels(
    train_water_levels
)

## Fit the fixed-order SARIMAX model

One SARIMAX is fitted, once, on the full training water-level series. The order is fixed and hand-picked, with a deliberately minimal daily seasonal term on top (P=1, D=0, Q=0, period 24). The training input is the interpolated series — the same one Auto-ARIMA receives, which keeps the two notebooks comparable.

**The arguments**

- `endog` (first positional) — `train_model_values`, the training series with internal gaps interpolated.
- `exog=train_exog` — the `(n_train, 6)` calendar Fourier matrix taken straight from the artifact's own columns. It must have one row per endogenous observation.
- `order=SARIMAX_ORDER` — the non-seasonal `(p, d, q)` described in Setup.
- `seasonal_order=SARIMAX_SEASONAL_ORDER` — the seasonal `(P, D, Q, m)` described in Setup.
- `.fit(disp=False)` — maximum-likelihood estimation via the Kalman filter. `disp=False` only silences the optimiser's per-iteration console output; it changes nothing about the fit.

Everything else stays at the statsmodels defaults, notably `trend=None` (no separate constant term, which with `d=1` and an exogenous block would not be identifiable anyway) and `enforce_stationarity` / `enforce_invertibility` left `True`, which keep the estimated parameters inside the stable region.

No validation split, no cross-validation, no held-out observations, no test rows and none of the other 47 engineered predictors take part in this fit.

The displayed table records the order, how many training rows there were, how many of them were reconstructed by interpolation, the exogenous columns used, and the fitted AIC. That AIC is only comparable against another model fitted on the *same* data — it says nothing about test accuracy, which is what the final cell measures.

In [ ]:
train_exog = train_features[CALENDAR_EXOG_COLUMNS].to_numpy()

sarimax_model = SARIMAX(
    train_model_values,
    exog=train_exog,
    order=SARIMAX_ORDER,
    seasonal_order=SARIMAX_SEASONAL_ORDER,
)
sarimax_result = sarimax_model.fit(disp=False)

model_configuration = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "model": "SARIMAX",
            "train_rows": len(train_water_levels),
            "interpolated_train_rows": interpolated_train_rows,
            "order": SARIMAX_ORDER,
            "seasonal_order": SARIMAX_SEASONAL_ORDER,
            "exog_columns": ", ".join(CALENDAR_EXOG_COLUMNS),
            "aic": sarimax_result.aic,
        }
    ]
)
print(f"SARIMAX fixed-order fit for {station_id}")
display(model_configuration)
display(pd.DataFrame([TRAIN_WATER_LEVEL_INTERPOLATION]))

## Issue rolling test forecasts

The parameters are frozen, but the *state* is not. Walking forward one hour at a time, each observed test water level (with its matching calendar row) is appended to the fitted filter, and a fresh 24-step forecast is issued from there.

This is what makes the evaluation realistic: at 14:00 a forecaster genuinely knows the 14:00 reading, and refusing to use it would understate the model. What they do not know are the readings after 14:00 — and nothing in this loop uses them.

The exogenous side needs one extra step compared with Auto-ARIMA. Each forecast requires calendar values for hours `t+1 … t+24`, which routinely run past the end of the test artifact, so `calendar_exog()` recomputes them from `future_timestamps` (a 24-hour `date_range` starting one hour after the issue time). This is legitimate precisely because calendar values are deterministic — knowing what time it will be tomorrow is not knowing the future.

**The arguments that make this leakage-free**

- `state.append([water_level], exog=test_exog[step:step + 1], refit=False)` — extends the fitted state with the newly observed hour. `refit=False` is the load-bearing argument: the coefficients stay exactly as estimated on the training split, and only the Kalman filter's state advances. With `refit=True` the model would re-estimate its parameters on data that includes test observations, and the reported score would be meaningless.
- A missing observation is appended as `NaN`. The Kalman filter treats it as an unobserved step and propagates its own prediction instead — no value is ever borrowed from a later hour to fill it.
- `get_forecast(steps=FORECAST_HORIZON_HOURS, exog=future_exog)` — `steps=24` produces exactly one value per lead hour, matching `TARGET_COLUMNS`, and `exog` must supply one row per requested step (a `(24, 6)` matrix) or statsmodels refuses to forecast. `.predicted_mean` takes the point forecast, i.e. the conditional mean, which is the quantity MAE and RMSE compare against.
- The shape check after each call turns a silent statsmodels shape change into an immediate failure.
- `np.vstack(all_test_predictions)[test_mask.to_numpy()]` — forecasts are issued at *every* test timestamp, because the filter has to walk through all of them in order to stay aligned, but only the cohort rows are kept for scoring.

In [ ]:
state = sarimax_result
test_timestamps = pd.DatetimeIndex(test_features["timestamp"])
test_exog = test_features[CALENDAR_EXOG_COLUMNS].to_numpy()

all_test_predictions: list[np.ndarray] = []
for step, (timestamp, water_level) in enumerate(zip(test_timestamps, test_water_levels)):
    state = state.append([water_level], exog=test_exog[step : step + 1], refit=False)
    future_timestamps = pd.date_range(
        timestamp + pd.Timedelta(hours=1), periods=FORECAST_HORIZON_HOURS, freq="h"
    )
    future_exog = calendar_exog(future_timestamps).to_numpy()
    forecast = np.asarray(
        state.get_forecast(
            steps=FORECAST_HORIZON_HOURS, exog=future_exog
        ).predicted_mean,
        dtype=float,
    )
    if forecast.shape != (FORECAST_HORIZON_HOURS,):
        raise RuntimeError(
            f"Expected {FORECAST_HORIZON_HOURS} forecast values; got {forecast.shape}"
        )
    all_test_predictions.append(forecast)

test_predictions = np.vstack(all_test_predictions)[test_mask.to_numpy()]

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort: aggregate MAE/RMSE, the same two metrics per lead time, and a short preview so the predictions can be eyeballed against their actual targets. There is no second pass and no refitting — what is printed here is the notebook's one and only result.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
print(f"SARIMAX test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))